<a href="https://colab.research.google.com/github/faizanahmed-ai/flyrank-ml-internship/blob/main/notebooks/03_working_with_the_full_release.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faizanahmed-ai/flyrank-ml-internship/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [1]:
%pip -q install duckdb huggingface_hub


In [ ]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [ ]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [ ]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY AND f.report_date > b.end_d - INTERVAL 60 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 60 DAY AND f.report_date > b.end_d - INTERVAL 90 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_60_90,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30,
               STDDEV(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY AND f.report_date > b.end_d - INTERVAL 60 DAY THEN f.gsc_avg_position END) AS pos_volatility_prior
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 90 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 750
    )
    SELECT * FROM windowed
""").df()
print(f'{len(features):,} content items with enough history')

**Threshold choice:** raised `imp_prev30` cutoff from 500 → 750. Rationale: at 500 the panel
includes very low-traffic pages where a 20% swing is 1-2 impressions of noise, not a real
signal. 750 trims the noisiest tail while keeping enough volume per client. Renamed
`imp_prev60` → `imp_60_90` since it is the isolated 60–90 day block, not a cumulative sum —
the old name was misleading.

## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [ ]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()

## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


**Objective:** Build a first predictive model and validate it honestly against a
simple baseline, before trusting any result.

**Label definition:** A content page is marked as "declining" if its impressions
dropped by more than 20% in the most recent 30 days compared to the prior 30 days.

**Why this avoids leakage:** The features used to predict (from the *prev-30*
window and query-mix data) are all built from information available *before* the
outcome window. The label itself is defined using the *last-30* window — the
outcome we're trying to predict. Because features and label come from non-overlapping
time periods, the model can't "see" the answer baked into its own inputs.

**Validation approach:**
- Split data into train (75%) and test (25%) sets, so evaluation happens on unseen data
- Compare model accuracy against a dumb baseline: always predicting the majority class
- A model is only meaningful if it clearly beats this baseline — not just matches it

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)
data['momentum_prior'] = data['imp_prev30'] / data['imp_60_90'].clip(lower=1)

feature_cols = ['imp_prev30', 'momentum_prior', 'pos_volatility_prior',
                'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

# --- Rule-based baseline: no ML, just momentum direction ---
baseline_pred = (model_data['momentum_prior'] < 1).astype(int)
print("--- Rule baseline (momentum_prior < 1) ---")
print(f'accuracy: {accuracy_score(y, baseline_pred):.3f}')
print(f'ROC-AUC:  {roc_auc_score(y, model_data["momentum_prior"] * -1):.3f}\n')

# --- Random split, 5-fold CV instead of single seed ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
aucs = []
for tr_idx, te_idx in skf.split(X, y):
    m = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)
    m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    aucs.append(roc_auc_score(y.iloc[te_idx], m.predict_proba(X.iloc[te_idx])[:, 1]))
print(f"--- Random split (5-fold CV) ---\nAUC range: {min(aucs):.3f} - {max(aucs):.3f}, mean: {np.mean(aucs):.3f}\n")

# --- Fit once more for full report + feature importance ---
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1).fit(X_tr, y_tr)
probs = model.predict_proba(X_te)[:, 1]
print("--- Random split (page-level), single seed for detail ---")
print(f'ROC-AUC: {roc_auc_score(y_te, probs):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))
print("\nFeature importance:")
print(pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False))

# --- Group split (client-level) — unchanged, keep as-is ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=model_data['client_hash_id']))
X_tr2, X_te2 = X.iloc[train_idx], X.iloc[test_idx]
y_tr2, y_te2 = y.iloc[train_idx], y.iloc[test_idx]
model2 = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1).fit(X_tr2, y_tr2)
probs2 = model2.predict_proba(X_te2)[:, 1]
print("\n--- Group split (client-level) ---")
print(f'ROC-AUC: {roc_auc_score(y_te2, probs2):.3f}')
print(classification_report(y_te2, model2.predict(X_te2), digits=3))

**Reading the precision, not just the AUC:** in the group split, precision on the
"declining" class is ~0.56 — meaning close to half the pages this model flags as declining
are false alarms. For a recommendation engine, this matters more than the AUC headline: it
sets the honest expectation that flagged pages need human review, not automatic action.
The rule baseline above (momentum direction only) is the floor this model needs to clear —
report both numbers side by side in the paper's Results section, not the RF number alone.

Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.
